In [2]:
!uv pip install opencv-python

Using Python 3.12.13 environment at: D:\Work\OldBack\Personal\Github Repository\ai_python_practice_questions\quetions\.venv
Resolved 2 packages in 1.82s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 3.26s
 + opencv-python==4.13.0.92


In [4]:
!uv pip install cv2-enumerate-cameras

Using Python 3.12.13 environment at: D:\Work\OldBack\Personal\Github Repository\ai_python_practice_questions\quetions\.venv
Resolved 1 package in 2.16s
Prepared 1 package in 1.24s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 269ms
 + cv2-enumerate-cameras==1.3.3


In [5]:
!uv pip show Pillow

Name: pillow
Version: 12.2.0
Location: D:\Work\OldBack\Personal\Github Repository\ai_python_practice_questions\quetions\.venv\Lib\site-packages
Requires:
Required-by: matplotlib


Using Python 3.12.13 environment at: D:\Work\OldBack\Personal\Github Repository\ai_python_practice_questions\quetions\.venv


In [13]:
import tkinter as tk # 编写程序
from cv2_enumerate_cameras import enumerate_cameras # 获取摄像头列表
import cv2 # 视觉库
from PIL import Image, ImageTk # 解决图像格式不兼容问题
import os, time

camera = None # 全局摄像头对象
current_frame = None # 当前帧 用于拍照
is_running = True # 控制刷新循环

def init_camera(cam_list, camera_label):
    selected = cam_list[0]
    cap = None

    for backend in (cv2.CAP_DSHOW, cv2.CAP_ANY):
        print(f"backend:{backend}")
        c = cv2.VideoCapture(selected.index, backend)
        if c.isOpened():
            cap = c
            break
        c.release()

    if cap is None:
        camera_label.config(text=f"打开失败：{selected.name}", fg="red")
        return None
    # ★修改4：强制 MJPG 编码，规避 DroidCam YUV 裸流被误当 BGR 解析变绿的问题
    # cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    camera_label.config(text=f"正在使用：{selected.name}", fg="green")
    return cap

# 实时画面更新
def update_frame(video_label):
    global camera, current_frame, is_running
    if camera is not None and camera.isOpened():
        ret, frame = camera.read()
        if ret:
            current_frame = frame.copy()
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            pil_img.thumbnail((640, 480), Image.Resampling.LANCZOS)
            imgtk = ImageTk.PhotoImage(pil_img, master=root)
            video_label.config(image=imgtk)
            video_label.last_image = imgtk
    if is_running:
        root.after(30, lambda: update_frame(video_label))

def take_photo():
    global current_frame
    save_dir = "captures"
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    file_name = f"{save_dir}/photo_{time.strftime("%Y%m%d_%H%M%S")}.png"
    cv2.imwrite(file_name, current_frame)
    print(f"拍照保存：{file_name}")  

def on_closing():
    global camera, is_running
    is_running = False
    if camera is not None:
        camera.release()
    root.destroy()

def main():
    global root, camera

    print("创建窗口")
    # 创建窗口
    root = tk.Tk()
    root.title("摄像工具")
    root.geometry("800x600")
    root.resizable(False, False)

    # 定义label标签
    camera_name_label_text = ""
    camera_label = tk.Label(root, text=camera_name_label_text, font=("Microsoft YaHei", 12), fg="blue")
    camera_label.pack(pady=5)

    video_label = tk.Label(root, bg='black')
    video_label.pack(pady=10, padx=10, expand= True, fill=tk.BOTH)

    cam_list = list(enumerate_cameras())
    if len(cam_list) != 0:
        camera = init_camera(cam_list ,camera_label)
        if camera:
            update_frame(video_label)

    else:
        camera_name_label_text = "未找到摄像头"
        camera_label.config(text=camera_name_label_text)

    # 定义button标签
    btn_frame = tk.Frame(root)
    btn_frame.pack(pady=10)

    capture_btn = tk.Button(btn_frame, text="拍照", command=take_photo, width=15, height=2, font=("Microsoft YaHei", 12))
    capture_btn.pack()

    # 启动 GUI主程序
    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.mainloop()

if __name__ == "__main__":
    main()

创建窗口
backend:700
backend:0
拍照保存：captures/photo_20260615_181507.png
拍照保存：captures/photo_20260615_181510.png
拍照保存：captures/photo_20260615_181515.png
拍照保存：captures/photo_20260615_181516.png
拍照保存：captures/photo_20260615_181517.png
拍照保存：captures/photo_20260615_181518.png
